# Kinnow ERP LLM: Kaggle runbook

This notebook is the control panel for training and scoring the Kinnow model on Kaggle's free GPUs.
Nothing here runs by itself. You run the cells you need, top to bottom.

**What Kinnow is.** An ERP assistant for GST, payroll and invoice matching. A Qwen2.5-7B-Instruct model is
fine-tuned (QLoRA) on reviewed Q&A, looks up reviewed rule notes before it answers, and sits behind a 4-layer
guardrail gateway (FastAPI). The plan it follows is `kinnow-production-plan-v1.md` in the repo.

**The target** (from the plan): overall 78%, GST 65%, Payroll 82%, Invoice 86%, false refusals under 3%.

**Where the code lives.** Everything real is in the GitHub repo `BonsAI-Agents/Kinnow_LLM`. This notebook only
downloads and starts the repo's scripts, so it stays short and the logic is version-controlled.

## Contents

| # | Section | Use it to |
|---|---|---|
| 1 | Before you start | Set up the Kaggle session correctly |
| 2 | Version history | See what each model version is, with its scores |
| 3 | v1 | Read the first run and why it failed |
| 4 | v2 | Read the pad-token fix and the first real scores |
| 5 | v3 | Read the lookup notes and generated calculation rows |
| 6 | v4 | Read Tejas's corrected data and the Section 50(3) change |
| 7 | v4 + answer features | Read what the hybrid lookup, calculators and sources line did |
| 8 | Run a job | Train a version, evaluate an adapter, check progress, read results |
| 9 | Troubleshooting | Find a known problem and its fix |
| 10 | Repo map | Find which file does what |
| 11 | Glossary | Look up a term |

## 1. Before you start

Do these once per session. A session ends after about 40 minutes without interaction, and **ending a session
deletes everything in `/kaggle/working`** (code, environment, result files). Adapters survive because training
pushes them to Hugging Face. Result files do not, so read or copy them out before the session ends.

| Setting | Value | Where |
|---|---|---|
| Accelerator | **GPU T4 x2** (about 15 GB each, no bfloat16) | Session options |
| Internet | **On** | Session options |
| Secret `HF_TOKEN` | A Hugging Face **write** token, attached to this notebook | Add-ons > Secrets |
| Weekly GPU quota | About 30 hours. A 7B training run costs about 4 hours, a full eval about 45 minutes | Kaggle account |

Rules this notebook follows:
- The token is read from Kaggle Secrets inside the helper below. It is never printed or written to a file.
- New adapters go to **private** Hugging Face repos, so an unreviewed model cannot be published by accident.
- Jobs run in the background and write a log. Closing the tab does not stop a job, but an idle session will end.

In [ ]:
# Helpers used by the rest of the notebook. Run this cell first in every session.
import os
import subprocess
import json

RAW = "https://raw.githubusercontent.com/BonsAI-Agents/Kinnow_LLM/main/scripts/"
W = "/kaggle/working"


def start_job(script, log="job.log", **env_vars):
    """Download scripts/<script> from GitHub and run it in the background. Returns the process.

    The Hugging Face token comes from Kaggle Secrets into the job's environment only.
    Extra keyword arguments become environment variables for the job, e.g. VERSION="v5"."""
    from kaggle_secrets import UserSecretsClient
    subprocess.run(f"curl -s {RAW}{script} -o {W}/{script}", shell=True, check=True)
    env = dict(os.environ, HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"), **env_vars)
    p = subprocess.Popen(f"python {W}/{script} > {W}/{log} 2>&1", shell=True, env=env, start_new_session=True)
    print(f"started {script} (pid {p.pid}); log: {W}/{log}")
    return p


def job_status(log="job.log"):
    """Milestones reached by the running job, plus the last line of its log."""
    path = f"{W}/{log}"
    if not os.path.exists(path):
        print("no log yet")
        return
    text = open(path, errors="replace").read()
    for mark in ("TRAIN DONE", "EVAL SAVED", "JOB DONE"):
        print(("done    " if mark in text else "pending ") + mark)
    if "Traceback" in text:
        print("ERROR: a Traceback is in the log; run show_log() to read it")
    last = text.replace("\r", "\n").strip().splitlines()[-1][:150] if text.strip() else ""
    print("last line:", last)


def show_log(log="job.log", lines=40):
    """The last lines of a job's log, without the long library warnings."""
    text = open(f"{W}/{log}", errors="replace").read().replace("\r", "\n").splitlines()
    text = [t for t in text if "warn" not in t.lower()]
    print("\n".join(t[:160] for t in text[-lines:]))



def show_results(log="job.log"):
    """The score table printed by the full eval (overall, per topic, Hindi, multi-turn, latency)."""
    text = open(f"{W}/{log}", errors="replace").read().splitlines()
    for i, line in enumerate(text):
        if "Full eval (" in line:
            print("\n".join(text[i:i + 17]))
            return
    print("no score table in the log yet")


def show_failures(results_json, limit=200):
    """One line per failing question: topic, tag, and the required terms the answer missed.
    results_json is a file like /kaggle/working/v5_full.json written by a job."""
    norm = lambda s: s.lower().replace(",", "")
    data = json.load(open(results_json))
    failed = [g for g in data["golden"] if not g["passed"] and g["category"] != "REFUSAL"]
    print(f"{len(failed)} failing questions")
    for g in failed[:limit]:
        missing = [w for w in g["must_contain"] if norm(w) not in norm(g["response"])]
        wrong = [w for w in g.get("must_not_contain", []) if norm(w) in norm(g["response"])]
        print(f"{g['category'][:3]} {g['tag'][:40]:<40} missing={missing} wrong={wrong}")

## 2. Version history

One row per trained model. "Score" is the overall pass rate on the 192 English golden questions
(`eval/golden_set.jsonl`), with the lookup on unless stated. **From v4 on, four Section 50(3) questions expect 18%
instead of 24%** (owner decision, approved by the CA per the owner), so v3 and v4 scores are compared on the same updated set
where it matters.

| Version | What changed | Adapter on Hugging Face (private) | Score | Section |
|---|---|---|---|---|
| Untrained | Qwen2.5-7B-Instruct as downloaded, no lookup | none | 22.9% | 4 |
| v1 | First QLoRA run. Answers never stopped (pad token bug) | not recorded (the trainer's default repo name is `KaraPatelDev/kinnow-v1`) | not recorded | 3 |
| v2 | Pad-token fix; Tejas's 500 rows plus our non-conflicting v1 rows | `KaraPatelDev/kinnow-v2-qwen25-sft-adapter` | 41.7%; 52.1% with lookup | 4 |
| v3 | Lookup notes; 312 generated calculation rows; Section 50(3) rows at 24% | `KaraPatelDev/kinnow-v3-qwen25-sft-adapter` | 54.7% with lookup (52.6% on the updated questions) | 5 |
| v4 | Tejas's corrected 500 rows plus 89 new Payroll and Invoice rows; Section 50(3) at 18% | `KaraPatelDev/kinnow-v4-qwen25-sft-adapter` | 50.0% with lookup | 6 |
| v4 + features | Same v4 model with hybrid lookup, calculators and a sources line (no retraining) | same as v4 | **57.8%** (52.6% without the sources line) | 7 |

**Reading the scores.** The eval has about 50 questions per topic, so each question is worth about 2 points.
Re-running the same eval can move individual items: v3 scored Hindi 55.6% on one run and 63.0% on an identical
re-run with the same overall score. Treat differences of a few points as noise unless they repeat.

### Full results ledger

Overall / GST / Invoice / Payroll, in percent.

| Run | Overall | GST | Invoice | Payroll | Notes |
|---|---|---|---|---|---|
| Untrained, no lookup | 22.9 | 1.9 | 5.8 | 14.0 | Baseline |
| v2, no lookup | 41.7 | 29.6 | 26.9 | 30.0 | |
| v2 + lookup (4 notes) | 52.1 | 42.6 | 40.4 | 42.0 | Lookup adds about 10 points |
| v3 + lookup (4 notes), old questions | 54.7 | 48.1 | 44.2 | 42.0 | Refusals 97.2, false refusals 0%, multi-turn 15%/50% |
| v3 + lookup (6 notes) | 53.6 | 40.7 | 44.2 | 46.0 | More notes did not help; 4 stays |
| v3 + new notes, updated questions | 52.6 | 46.3 | 36.5 | 44.0 | Same questions as v4 |
| v4 + lookup (4 notes) | 50.0 | 38.9 | 38.5 | 42.0 | Refusals 94.4; within noise of v3 |
| v4 + hybrid lookup + calculators | 52.6 | 38.9 | 40.4 | 48.0 | Without the sources line |
| v4 + hybrid + calculators + sources line | **57.8** | 44.4 | 50.0 | 52.0 | Refusals 97.2, false refusals 0%, multi-turn 15%/53% |

Fact check against Tejas's 50 claims (`eval/fact_checks.jsonl`): with lookup v3 48/50, v4 48/50; without lookup
v3 38/50, v4 34/50.

## 3. v1: the first run (historical)

**What it was.** The first QLoRA fine-tune of Qwen2.5-7B-Instruct on our own reviewed Q&A (`data/v1_training`).
Settings, as in the later versions: 4-bit nf4, r=64, alpha=128, fp16 (a T4 has no bfloat16), completion-only loss,
600 steps, effective batch 8, learning rate 2e-4 with a cosine schedule.

**What went wrong.** The padding token was the same as the end-of-turn token (`<|im_end|>`). Padding is masked out
of the loss, so the model was never taught to stop. Answers ran on and rambled, and the scores were meaningless.

**The fix** (in v2): a distinct pad token (`<|endoftext|>`), plus a guard in `training/sft_train.py`
(`check_end_token_trained`) that fails the run if the end token is not being trained.

**Kept for reference.** `data/v1_training/raw.jsonl` still feeds the guardrail's category centroids in CI, so
it must not be edited. No v1 scores are recorded here.

## 4. v2: pad-token fix and Tejas's 500 rows

**Data.** `scripts/merge_tejas_data.py` merged Tejas's 500 reviewed rows with our v1 rows that do not conflict.
Rules: Tejas's rows win on a duplicate prompt; all refusal rows are kept; v1 rows that rely on the old income-tax
regime (Income-tax Act 1961, Form 16/24Q/26AS, old TDS sections) are set aside in `dropped_old_regime.jsonl`
because the new Act (Sections 392/393, Forms 130/138) replaced them.

**Training.** Same settings as v1 with the pad-token fix. About 25 seconds per step on 2 x T4 at the start.
The first evaluation crashed with CUDA out-of-memory because the default eval batch size is 8; it is now 1.

**Results.**

| Run | Overall | GST | Invoice | Payroll |
|---|---|---|---|---|
| Untrained Qwen2.5-7B | 22.9 | 1.9 | 5.8 | 14.0 |
| v2 | 41.7 | 29.6 | 26.9 | 30.0 |
| v2 + lookup | 52.1 | 42.6 | 40.4 | 42.0 |

**What we learned.**
- Training adds about 19 points over the untrained model; the lookup adds about 10 more.
- Answers became short (about 41 words) and stopped correctly, at about 6 seconds each.
- The remaining failures were missing exact citations and figures (a knowledge gap), not a scoring bug.

**The lookup** (`guardrails/retrieval.py`) puts the 4 most relevant reviewed Q&A notes in front of the question.
Notes are built by `scripts/build_kb.py` from the training data, without refusal rows, generated rows, or any
question that also appears in the eval files.

## 5. v3: lookup notes plus generated calculation rows

**Data changes.**
- 312 calculation rows computed in Python by `scripts/generate_calc_rows.py` (gratuity 15/26, EPF 12% on the
  Rs 25,000 ceiling, ESI 0.75%/3.25% rounded up, new-regime tax with the Rs 75,000 deduction and 87A rebate,
  leave encashment, interest, Rule 37 and Rule 42, e-invoice thresholds, GST TDS, composition rates).
  Every figure is computed, so the arithmetic cannot be wrong; the rules come from Tejas's checklist.
- Section 50(3) interest rows released as "assumed confirmed" at **24%** on the owner's decision (replaced by 18% in v4, see section 6).
  They are listed in `data/v2_training/assumed_pending_ca.jsonl`.
- `eval/fact_checks.jsonl` and `eval/run_fact_check.py`: 50 claims from Tejas's `FACTS_TO_VERIFY.md`, scored with
  all_of / any_of / must_not words.

**Results** (with lookup of 4 notes, old questions): overall 54.7, GST 48.1, Invoice 44.2, Payroll 42.0, refusals
97.2, false refusals 0%, multi-turn 15%/50%. Fact check 48/50 with lookup, 38/50 without.
Payroll did not move even with 150 generated Payroll rows: the eval's Payroll questions are mostly multi-condition
cases, not the plain formulas the rows cover.

**Retrieval size test.** 6 notes per question scored 53.6, lower than 4 (54.7), so the lookup stays at 4.

## 6. v4: Tejas's corrected data and the Section 50(3) change

**Data changes** (merged by `scripts/merge_tejas_data.py`, 2,691 rows in total).
- `data/tejas/kinnow_sft_500_v2.jsonl`: Tejas's 500 rows with Section 50(3) corrected to 18%, and the
  meal-voucher row replaced by a PF-overtime row.
- `data/tejas/kinnow_sft_payroll_invoice_v2.jsonl`: 89 new worked examples (44 Payroll, 45 Invoice) with traps
  (Rs 1 thresholds, mid-year joiners, late credit notes, rate changes) and exact section/rule/form names.
- Our own v1 rows that state 24% for Section 50(3) are held out (`held_for_verification.jsonl`).
- The generator's interest template now writes 18%, and four golden questions expect 18%.

**The 18% decision.** Tejas says Section 50(3) interest is 18% (Notification 14/2022-CT) and that the Act's 24%
is only a ceiling. The owner chose 18% on 21 Nov 2026 and, per the owner, the CA approved it on 22 Nov 2026
(logged in `data/tejas_review/DECISIONS.md`; no written CA note is stored in the repo yet, so add it there).
An independent web check found secondary sources still stating 24% and read Notification 14/2022-CT as the one that
introduced Rule 88B (the calculation method); the official text was not read. If the notification or the CA's note
ever says otherwise, change the generator template, the four golden questions and the fact check, then retrain.

**Training.** 600 steps, about 3 h 50 min (including the checkpoint evaluations). Validation loss fell at every
checkpoint: 1.40, 1.23, 1.10, 1.05, 1.02, 1.01.

**Results** (lookup of 4 notes, updated questions): overall 50.0, GST 38.9, Invoice 38.5, Payroll 42.0,
refusals 94.4, false refusals 0%, Hindi 55.6, multi-turn 10%/50%.
Fact check 48/50 with lookup, 34/50 without.

**v3 against v4 on identical questions and notes:** v3 52.6, v4 50.0. That is about 5 questions out of 192, within
the normal noise of this eval. The 589 new Tejas rows did not move the score measurably.

## 7. v4 + answer features (no retraining)

Three changes to how answers are produced. Each has an eval switch (`--hybrid`, `--calc`, `--cite`).

| Feature | What it does | File |
|---|---|---|
| Hybrid lookup | Word matching plus embedding similarity, merged by reciprocal rank fusion. The top 4 notes hold the needed terms for 70% of eval questions, against 64% before | `guardrails/retrieval.py` (`HybridRetriever`) |
| Calculators | Gratuity, PF, ESI, new-regime tax and interest are computed in code and given to the model as "Computed by the system". Silent unless the question has exactly one amount and no special wording. They apply to only 5 of 146 eval questions | `guardrails/calculators.py` |
| Sources line | After the answer, adds "Sources: Section 16(2)(aa), Rule 37, ..." taken only from the retrieved notes | `guardrails/retrieval.py` (`sources_line`) |

**Result on the v4 model** (same questions):

| Setup | Overall | GST | Invoice | Payroll |
|---|---|---|---|---|
| v4, old lookup | 50.0 | 38.9 | 38.5 | 42.0 |
| + hybrid lookup + calculators | 52.6 | 38.9 | 40.4 | 48.0 |
| + sources line | **57.8** | 44.4 | 50.0 | 52.0 |

**How to read this.**
- Hybrid lookup plus calculators gave about +2.6; the sources line gave about +5.2.
- The sources line adds accurate citations but does not change the model's reasoning, and part of its gain is the
  scorer rewarding exact wording. The lookup variant was chosen using the eval's required terms, so the numbers
  are probably optimistic.
- The only fair measure of real accuracy is a human review (Tejas or a CA) of a sample of answers. That is the
  next step before deciding about the 78% target.

**Now in the server.** `guardrails/answer_support.py` wires all three features into the gateway (on by default,
each with an environment switch, each falling back to the plain behaviour on any error). It adds about 0.1 second
per request on a laptop CPU for the lookup. Build the knowledge file and its embeddings with
`python scripts/merge_tejas_data.py && python scripts/build_kb.py --embed`, then mount `data/knowledge/` next to
`data/centroids.npz`. One known weakness: the sources line sometimes cites a loosely related note (for example
Section 43B(h) on a question about Rule 37), because it takes citations from the two best notes.

**Human review (next step).** `data/review/review_sheet_for_tejas.csv` holds 50 answers in random order: 40 that the
scorer failed and 10 that passed only because of the sources line. The scorer's verdicts are hidden from the reviewer
(they are in `review_KEY_do_not_share.csv`). Tejas marks each CORRECT, PARTLY or WRONG. Compare his verdicts with the
scorer's to learn the real accuracy and how much of the gap to 78% is wording.

## 8. Run a job

Run the helper cell in section 1 first. Each job is one background script from the repo. Jobs write a log in
`/kaggle/working`, and their scripts print milestone markers (`TRAIN DONE`, `EVAL SAVED`, `JOB DONE`).

| I want to | Cell | Time |
|---|---|---|
| Train a new version and score it | 8a | about 5 hours |
| Score an existing adapter | 8b | about 15 min setup + 45 min |
| See how a running job is doing | 8c | instant |
| Read the scores | 8d | instant |
| See which questions fail and why | 8e | instant |

### 8a. Train a new version

Set `VERSION` to a new label. The job rebuilds the data, trains 600 steps, pushes a private adapter
`KaraPatelDev/kinnow-<VERSION>-qwen25-sft-adapter`, then runs the full eval.
Optional settings: `SEED` (to measure run-to-run variation), `MAX_STEPS`, `FLAGS` (eval switches),
`BASE` (base model). Training uses most of a weekly GPU quota; do not start it casually.

In [ ]:
# 8a. Train a new version, then score it. Change VERSION first.
train_job = start_job("kaggle_train.py", log="train.log", VERSION="v5")

### 8b. Score an existing adapter

Use this to re-score a version, or to test a feature switch without retraining.
`FLAGS` are the eval switches: `--rag 4` (lookup of 4 notes), `--hybrid` (embedding lookup), `--calc`
(calculators), `--cite` (sources line; the eval also reports the score without it).
The result file is `/kaggle/working/<TAG>_full.json`.

In [ ]:
# 8b. Score an existing adapter. Change ADAPTER, TAG and FLAGS as needed.
eval_job = start_job(
    "kaggle_eval.py", log="job.log",
    ADAPTER="KaraPatelDev/kinnow-v4-qwen25-sft-adapter",
    TAG="v4-check",
    FLAGS="--rag 4 --hybrid --calc --cite",
)

### 8c. Check progress

Training logs a progress bar per step and a validation loss every 100 steps. The eval prints nothing until it
finishes. If the status shows an ERROR, run `show_log()` to read it.

In [ ]:
# 8c. How is the job doing?  (use log="train.log" for a training job)
job_status("job.log")

### 8d. Read the scores

Prints the table from the finished eval: overall, per topic, refusals, false refusals, Hindi, multi-turn, latency.

In [ ]:
# 8d. Scores from the finished eval.
show_results("job.log")

### 8e. See which questions fail and why

Prints one line per failing question with the terms the answer missed. Use it to tell a wording miss
(the right answer without the exact phrase) from a real mistake. Read a few full answers before trusting a pattern.

In [ ]:
# 8e. Failing questions for a finished run. Change the file name to the run you want.
show_failures("/kaggle/working/v4-check_full.json")

## 9. Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Answers never stop, scores near zero | Pad token equal to the end-of-turn token, so the end is never trained | Fixed in `training/sft_train.py` (distinct pad token, plus a guard that fails the run). Do not remove the guard |
| CUDA out-of-memory at the first evaluation (step 100) | Default eval batch of 8 long sequences needs over 3 GiB of logits | Eval batch size is 1 in the trainer |
| Training crashes or warns about `bf16` | A T4 has no bfloat16 | The trainer uses fp16 on T4 |
| `pip install` fails on the pinned training stack | Kaggle's default Python is 3.13 with newer torch | The jobs build a Python 3.11 environment in `/kaggle/working/venv` with `uv` |
| `SyntaxError` in a script on Kaggle but not locally | Quoting inside f-strings that only Python 3.12+ allows | Use single quotes inside f-strings; the venv is 3.11 |
| Everything in `/kaggle/working` is gone | The session ended after about 40 minutes of no interaction, or after 12 hours | Re-run section 1, then the job. Adapters are safe on Hugging Face. Copy result files out early |
| `Traceback ... CalledProcessError` right after a score table | The eval exits with an error code whenever a target is missed. This is expected | The job scripts ignore that exit code; read the score table above the traceback |
| Hindi or multi-turn score moves between identical runs | Those sets are small (27 and 20 items) and GPU generation is not bit-exact | Treat one or two items as noise |
| "Failed to save draft" in the notebook header | A temporary save error in the Kaggle editor | Harmless; it clears itself |
| Page freezes when pasting very long single lines | Kaggle editor | Keep cells short; long logic lives in `scripts/` |
| `HF_TOKEN` error when starting a job | The secret is not attached to this notebook | Add-ons > Secrets, attach `HF_TOKEN` |
| Repo is public | It was made public only for Kaggle | Make it private again when told to; then the `raw.githubusercontent.com` download needs a token |

## 10. Repo map

| Path | What it does |
|---|---|
| `scripts/kaggle_train.py` | Kaggle job: set up, rebuild data, train, push adapter, score it |
| `scripts/kaggle_eval.py` | Kaggle job: set up, rebuild notes, score an existing adapter |
| `scripts/merge_tejas_data.py` | Builds `data/v2_training/raw.jsonl` from Tejas's files, our v1 rows and the generated rows; sets aside old-regime rows and rows held for review |
| `scripts/split_data.py` | Splits the merged data 90/10 into train and validation |
| `scripts/generate_calc_rows.py` | Computes the 312 calculation rows |
| `scripts/build_kb.py` | Builds the lookup notes `data/knowledge/kb.jsonl` (training data without refusals, generated rows or eval questions) |
| `training/sft_train.py` | QLoRA fine-tune (4-bit, r=64, completion-only loss, 600 steps); env `SEED`, `MAX_STEPS`, `HF_REPO` |
| `training/COMPARE_MODELS.md` | How to compare two base models fairly (Qwen2.5-7B vs Qwen3-8B) |
| `eval/run_full_eval.py` | Full eval: 192 golden questions, Hindi set, multi-turn set, false-refusal rate; switches `--rag --hybrid --calc --cite` |
| `eval/run_fact_check.py` | Checks 50 of Tejas's claims; `--notes-only` needs no GPU |
| `eval/golden_set.jsonl` | The 192 English eval questions (GST, Payroll, Invoice, Refusal) |
| `guardrails/retrieval.py` | The rule lookup: word matching, hybrid matching, sources line |
| `guardrails/calculators.py` | The deterministic calculators |
| `guardrails/pipeline.py` | The 4-layer guardrail gateway used in production. It takes an optional `AnswerSupport` |
| `guardrails/answer_support.py` | Wires the lookup, calculators and sources line into the gateway; switches `KINNOW_LOOKUP`, `KINNOW_CALC`, `KINNOW_SOURCES`, `KINNOW_LOOKUP_K`, `KINNOW_KB_PATH` |
| `data/review/` | Local only (not in git): the 50-answer human review sheet for Tejas and its answer key |
| `data/tejas/`, `data/tejas_review/` | Tejas's reviewed data and his facts checklist (`FACTS_TO_VERIFY_v2.md`) |
| `data/v2_training/` | The current training set and the files listing held or assumed rows |
| `kinnow-production-plan-v1.md` | The governing plan and its targets |

## 11. Glossary

| Term | Meaning |
|---|---|
| QLoRA | Fine-tuning a model with small trainable add-on weights on top of a frozen 4-bit copy, so a 7B model trains on two T4s |
| SFT | Supervised fine-tuning: training on question and answer pairs |
| Adapter | The small set of trained weights QLoRA produces; it is loaded on top of the base model |
| Completion-only loss | The model is graded on the answer tokens only, not on the question |
| Lookup (RAG) | Finding reviewed Q&A notes for a question and showing them to the model before it answers |
| Hybrid lookup | Lookup that combines word matching with embedding similarity |
| Sources line | A line like "Sources: Section 16(2)(aa), Rule 37" added after the answer, taken from the retrieved notes |
| Golden set | The 192 fixed English questions each model is scored on; an answer passes if it has the required words |
| Fact check | 50 claims from Tejas's checklist; each passes if the answer has the right figure or word |
| False refusal | Refusing a legitimate ERP question. The limit is 3% |
| Held / assumed rows | Training rows waiting for a CA: held ones are left out, assumed ones are used on the owner's decision |
| Noise | Score movement that happens by chance. With about 50 questions per topic, one question is about 2 points |